In [ ]:
!pip install langchain-groq langchain==0.3.25 duckduckgo-search langchain_community ddgs langchain-community  --quiet

In [18]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory
import getpass

In [ ]:
api_key = getpass.getpass("Enter your Groq API key: ")

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=api_key
)

In [ ]:
!pip install --upgrade langchain-core langchain-groq langchain

In [ ]:
single_turn_prompt = "Tell me about India"
print(llm.invoke(single_turn_prompt).content)

In [ ]:
single_turn_prompt_new = "Tell me what is its capital?"
print(llm.invoke(single_turn_prompt_new).content)

In [ ]:
structured_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Provide a brief explanation of {topic} and list its three main components."
)

chain = structured_prompt | llm
print(chain.invoke({"topic": "Gravity"}).content)

In [22]:
conversation = ConversationChain(
    llm=llm,
    verbose=False,
    memory=ConversationBufferMemory()
)

print(conversation.predict(input="How about Walmart?"))
print(conversation.predict(input="Where is it headquartered?"))

Walmart is an American multinational retail corporation that was founded on July 2, 1962, by Sam Walton in Rogers, Arkansas. It has grown to become the largest retail corporation in the world, with over 12,400 stores in 27 countries and a workforce of more than 2.3 million employees. Walmart's revenue for 2022 was approximately $572 billion, making it one of the largest companies in the world by revenue.
Walmart is headquartered in Bentonville, Arkansas. Specifically, the corporate office of Walmart is located at 702 S.W. 8th Street, Bentonville, AR 72716. This address has been the official headquarters of the company since its founding in 1962, and it remains the central hub for Walmart's global operations and decision-making processes.


In [23]:
print(conversation.predict(input="Where is it headquartered?"))

You're asking where Walmart is headquartered again. I'll give you the same answer: Walmart is headquartered in Bentonville, Arkansas. Specifically, the corporate office of Walmart is located at 702 S.W. 8th Street, Bentonville, AR 72716. This address has been the official headquarters of the company since its founding in 1962, and it remains the central hub for Walmart's global operations and decision-making processes.


**Prompt Template**

In [24]:
basic_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms for a beginner."
)


In [25]:
topic_prompt = basic_template.format(topic="machine learning")
print("Generated Prompt:", topic_prompt)
print("\nResponse:")
response = llm.invoke(topic_prompt)
print(response.content)

Generated Prompt: Explain machine learning in simple terms for a beginner.

Response:
Machine learning is a branch of artificial intelligence (AI) that enables computers to learn from data, make decisions, and improve their performance over time. Here's a simple explanation:

**Imagine a Child Learning**

Think of machine learning like a child learning to ride a bike. At first, the child doesn't know how to ride and will probably fall off. But with practice and guidance, the child learns to balance, steer, and ride the bike. The child is learning from their experiences and making improvements each time they ride.

**How Machine Learning Works**

Machine learning is similar. We feed a computer large amounts of data, and it uses this data to make predictions or decisions. The computer learns from the data and improves its performance over time, just like the child learning to ride a bike.

Here are the basic steps:

1. **Data Collection**: We gather a large dataset, such as images, text,

In [26]:
advanced_template = PromptTemplate(
    input_variables=["subject", "audience", "tone", "length"],
    template="""
You are an expert educator. Create a {length} explanation about {subject}
for {audience}. Use a {tone} tone and include practical examples.

Topic: {subject}
Target Audience: {audience}
Tone: {tone}
Length: {length}

Your explanation:
"""
)

In [27]:
advanced_prompt = advanced_template.format(
    subject="quantum computing",
    audience="High School Students",
    tone="Professional",
    length="100 words"
)
print("Generated Prompt:", advanced_prompt)
print("\nResponse:")
print(llm.invoke(advanced_prompt).content)

Generated Prompt: 
You are an expert educator. Create a 100 words explanation about quantum computing
for High School Students. Use a Professional tone and include practical examples.

Topic: quantum computing
Target Audience: High School Students
Tone: Professional
Length: 100 words

Your explanation:


Response:
Quantum computing is a revolutionary technology that harnesses the principles of quantum mechanics to perform calculations exponentially faster than classical computers. In a classical computer, information is represented as 0s and 1s, whereas in a quantum computer, it's represented as qubits, which can exist in multiple states simultaneously. This allows quantum computers to process vast amounts of data in parallel, making them ideal for complex problems like cryptography, optimization, and simulation. For instance, Google's quantum computer, Bristlecone, can simulate a molecule in a matter of minutes, which would take a classical computer thousands of years to accomplish.


**Zeroshot prompting**

In [28]:
def create_chain(prompt_template):
    """
    Create a LangChain chain with the given prompt template.

    Args:
        prompt_template (str): The prompt template string.

    Returns:
        LLMChain: A LangChain chain object.
    """
    prompt = PromptTemplate.from_template(prompt_template)
    return prompt | llm

In [29]:
direct_task_prompt = """Classify the sentiment of the following text as positive, negative, or neutral.
Do not explain your reasoning, just provide the classification.

Text: {text}

Sentiment:"""

direct_task_chain = create_chain(direct_task_prompt)

# Test the direct task specification
texts = [
    "The product met my expectation. Great value for money",
    "The weather today is quite typical for this time of year.",
    "I wish the restaurant wait times were shorter. We wasted a lot of time"
]

for text in texts:
    result = direct_task_chain.invoke({"text": text}).content
    print(f"Text: {text}")
    print(f"Sentiment: {result}")

Text: The product met my expectation. Great value for money
Sentiment: Positive
Text: The weather today is quite typical for this time of year.
Sentiment: Neutral
Text: I wish the restaurant wait times were shorter. We wasted a lot of time
Sentiment: Negative


In [30]:
multistep_reasoning_prompt = "Explain the steps involved in solving the {math_problem} and then solve the math problem step by step"
multistep_reasoning_chain = create_chain(multistep_reasoning_prompt)

problem ="3*4+(6-4)"
result = multistep_reasoning_chain.invoke({"math_problem": problem}).content
print(f"Problem: {problem}")
print(f"Result: {result}")

Problem: 3*4+(6-4)
Result: To solve the math problem 3*4+(6-4), we need to follow the order of operations (PEMDAS):

1. **P**arentheses: Evaluate expressions inside the parentheses first.
2. **E**xponents: Evaluate any exponential expressions next (none in this case).
3. **M**ultiplication: Evaluate any multiplication operations after that.
4. **D**ivision: Evaluate any division operations after multiplication.
5. **A**ddition: Evaluate any addition operations after that.
6. **S**ubtraction: Finally, evaluate any subtraction operations.

Now, let's apply these steps to the given problem:

1. Evaluate expressions inside the parentheses: 
   Inside the parentheses we have (6-4)
   So, 6-4 = 2

   The problem now looks like this: 3*4+2

2. Evaluate multiplication:
   Multiply 3 and 4.
   3*4 = 12

   The problem now looks like this: 12+2

3. Evaluate addition:
   Add 12 and 2.
   12 + 2 = 14

Therefore, the solution to the problem 3*4+(6-4) is 14.


**DsPY to control output**

In [31]:
!pip install dspy-ai

2256.35s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  Using cached regex-2026.1.15-cp312-cp312-macosx_11_0_arm64.whl.metadata (40 kB)
  Using cached fastuuid-0.14.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (1.1 kB)
  Using cached importlib_metadata-8.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached tiktoken-0.12.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.7 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-0.30.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached zipp-3.23.0-py3-none-any.whl.metadata (3.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 27.3 MB/s  0:00:00 eta 0:00:01
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached fastuuid-0.14.0-cp312-cp312-macosx_11_0_arm64.whl (251 kB)
Using cached

In [32]:
import dspy
lm = dspy.LM(
    model="groq/llama-3.1-8b-instant",
    api_key=api_key,
    max_tokens=600
)

dspy.settings.configure(lm=lm)


In [33]:
class CityInfo(dspy.Signature):
    """Return structured facts about a city in JSON format."""
    city = dspy.InputField(desc="Name of the city")
    output = dspy.OutputField(
        desc="JSON with keys: history, culture, food"
    )

In [34]:
city_info_module = dspy.Predict(CityInfo)


In [35]:
response = city_info_module(city="London")
print(response.output)

{
  "history": "London is one of the oldest cities in Europe, dating back to Roman times. The Romans established the city as Llundain in 43 AD. Throughout history, London has experienced numerous invasions and conflicts, with the Normans taking control in 1066. Today, the city is a bustling metropolis with a rich cultural and historical heritage.",
  "culture": "London is known for its vibrant cultural scene, with numerous museums, galleries, and performance venues. The city is home to the British Museum, the National Gallery, and the Tate Modern. The West End is a major hub for theater and music, with numerous productions and performances throughout the year. London also hosts several major festivals, including the Notting Hill Carnival and the London Festival.",
  "food": "London is famous for its traditional British cuisine, including fish and chips, roast beef, and full English breakfasts. The city is also home to a diverse range of international cuisines, including Chinese, Indian

In [37]:
cities = ["Tokyo", "Mumbai", "London","Paris"]

for city in cities:
    print(f"\n--- Information about {city} ---")
    response = city_info_module(city=city)
    print(response.output)


--- Information about Tokyo ---
{
  "history": "Tokyo has a rich history dating back to the 15th century, having been a major commercial center and capital of Japan. The city has been influenced by various cultures, including Japanese, Chinese, and Western.",
  "culture": "The culture of Tokyo is known for its vibrant and eclectic mix of traditional and modern elements. From ancient temples and shrines to cutting-edge technology and fashion, the city offers a unique blend of old and new.",
  "food": "Tokyo is famous for its diverse and sophisticated food scene, offering a wide range of traditional Japanese cuisines, such as sushi, ramen, and tempura, as well as modern fusion restaurants and high-end dining options."
}

--- Information about Mumbai ---
{
  "history": {
    "founding": "1653",
    "former names": "Bombay",
    "historical significance": "Historically, Mumbai was a major trading center due to its natural harbor and access to the Arabian Sea. Its strategic location made i

**Few Shot Prompting**

In [ ]:
# Few-shot prompting example for sentiment classification in e-commerce

few_shot_prompt_template = """Classify the sentiment of the following customer product review
as positive, negative, or neutral.

Here are a few examples:

Review: The shoes fit perfectly and the quality is amazing.
Sentiment: positive

Review: The delivery was late and the product was damaged.
Sentiment: negative

Review: The packaging was okay, nothing special.
Sentiment: neutral

Review: {review}
Sentiment:
Answer concisely in 1 or two words
"""

# Assuming you have a helper function that wraps the LLM call
few_shot_chain = create_chain(few_shot_prompt_template)

# Test the few-shot prompting with sample product reviews
reviews_to_classify = [
    "The headphones have excellent sound quality, totally worth the price!",
    "I'm not happy, the phone case broke after two days of use.",
    "The product is fine, but I don’t feel strongly about it either way."
]

for review in reviews_to_classify:
    result = few_shot_chain.invoke({"review": review}).content
    print(f"Review: {review}")
    print(f"{result}\n")


**Chain of Thought Prompting**

In [ ]:


# Chain of Thought prompt
cot_prompt = PromptTemplate(
    input_variables=["question"],
    template="Answer the following question step by step concisely: {question}"
)

# Create chains

cot_chain = cot_prompt | llm

# Example question
question = """A cylindrical water tank with a radius of 1.5 meters and a height of 4 meters is 2/3 full.
If water is being added at a rate of 10 liters per minute, how long will it take for the tank to overflow?
Give your answer in hours and minutes, rounded to the nearest minute.
(Use 3.14159 for π and 1000 liters = 1 cubic meter)"""

# Get responses

cot_response = cot_chain.invoke(question).content


print("\nChain of Thought Response:")
print(cot_response)

**Prompt Chaining**

In [ ]:
from langchain_core.runnables import RunnableLambda

# Define prompt templates and chains
prompts = {
    "summarize": ("review", "Summarize the following customer review into a single sentence:\n\nReview: {review}\n\nSummary:"),
    "sentiment": ("summary", "Classify the sentiment of the following review summary as positive, negative, or neutral:\n\nSummary: {summary}\n\nSentiment:"),
    "suggest": ("sentiment", "Based on the following sentiment, provide a brief suggestion for the seller:\n\nSentiment: {sentiment}\n\nSuggestion:")
}

chains = {}
for key, (var, template) in prompts.items():

    prompt = PromptTemplate(input_variables=[var], template=template)
    chains[key] = prompt | llm

# Helper to extract content
def extract_content(x):
    return x.content if hasattr(x, "content") else str(x)

# Helper to wrap output into dict for next chain
def wrap_output(key):
    return lambda x: {key: extract_content(x)}

# Compose full chain
full_chain = (
    chains["summarize"]
    | RunnableLambda(wrap_output("summary"))
    | chains["sentiment"]
    | RunnableLambda(wrap_output("sentiment"))
    | chains["suggest"]
)

# Example review
review = "The product arrived late and was slightly damaged. I am very disappointed with the quality and the delivery service. I would not recommend this product."

# Run chain
result = full_chain.invoke({"review": review})
suggestion = extract_content(result)

print(f"Review:\n{review}\n")
print(f"Suggestion:\n{suggestion}")


**System Message & Human Message**

In [40]:
from langchain_core.messages import SystemMessage, HumanMessage

def translate_to_hindi(human_message_content: str):
    """
    Translates an English sentence to Hindi using the language model.

    Args:
        human_message_content: The English sentence to translate.
    """
    messages = [
        SystemMessage(content="You are a helpful assistant that translates English to Hindi. Just translate the given sentence. Do not answer the question"),
        HumanMessage(content=human_message_content)
    ]
    response = llm.invoke(messages)
    print(response.content)


In [41]:
translate_to_hindi("Delhi has Qutub Minar")

दिल्ली में कुतुब मीनार है।
